In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.17 The Hydrogen Atom

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.17",
    title="The Hydrogen Atom",
    blurb="The triumph. Put the Coulomb attraction of a proton for an electron into "
    "the radial equation, and out come the energies that explain the hydrogen "
    "spectral lines, the orbitals that shape chemistry, and a surprise: the energy "
    "depends only on the principal quantum number, not on angular momentum, so the "
    "s, p, and d states of a shell all share one energy. That extra degeneracy is no "
    "accident — it is the signature of a hidden symmetry, the same one that keeps the "
    "planets' orbits from precessing.",
    difficulty="advanced",
    estimate="180–220 min",
)

## Notebook overview

This is the notebook the whole volume has been building toward. Every tool is now in hand — the
eigenvalue solver of [§6.10](schrodinger-on-a-computer.ipynb), the spherical harmonics of [§6.15](orbital-angular-momentum.ipynb), and the radial reduction of [§6.16](central-potentials-3d.ipynb) — and we
point all of them at the one potential that matters most: the **Coulomb attraction** of a proton for an
electron, $V(r)=-Z/r$. What comes out is the **hydrogen atom**, the first real atom ever solved exactly,
and the calculation that turned quantum mechanics from a promising idea into the established theory of
matter.

It delivers two triumphs. The first is the **Rydberg spectrum** $E_n=-13.6\,Z^2/n^2\,$eV: feeding
$V=-Z/r$ into the radial equation of [§6.16](central-potentials-3d.ipynb) yields bound states whose energies depend on a single integer
$n$, and whose *differences* are exactly the spectral lines — Lyman, Balmer, Paschen — that
spectroscopists had catalogued for decades without understanding. Bohr had guessed this formula in 1913
from a model he knew was provisional; here Schrödinger's equation produces it exactly, on a grid, in
seconds. The second triumph is the **orbitals** $\psi_{n,l,m}=R_{n,l}(r)Y_l^m(\theta,\varphi)$ — the
1s, 2s, 2p, 3d shapes that organize all of chemistry, labelled by three quantum numbers: $n$ (the shell),
$l$ (angular momentum, $0\le l\le n-1$), and $m$ (orientation).

And then the surprise. The Coulomb energy depends **only on $n$, not on $l$**: the 2s and 2p states,
though shaped completely differently, have *exactly* the same energy. Rotational symmetry alone would
only make the $2l+1$ orientations degenerate; this extra "accidental" $l$-degeneracy is special to the
$1/r$ potential, and it is no accident at all. It is the fingerprint of a **hidden symmetry** — the
conservation of the quantum **Runge–Lenz vector**, which enlarges the rotation group $SO(3)$ to $SO(4)$,
the very same conserved quantity that keeps a classical Kepler orbit from precessing (the thread from
Volumes I, II, and IV, where Mercury's *failure* to close revealed relativity). Counting a shell's states
gives $n^2$ orbitals, $2n^2$ with spin — exactly the $2,8,18,32$ capacities of the periodic table's rows.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the [§6.16](central-potentials-3d.ipynb) radial reduction $u=rR$ with the Coulomb $V_{\text{eff}}
=-Z/r+l(l+1)/2r^2$, the efficient `scipy.linalg.eigh_tridiagonal` for the radial eigenproblem, and
`scipy.special.genlaguerre` for the analytic hydrogen radial functions in the comparison.

> **Units and method notes.** We work in **atomic units** ($\hbar=m_e=e=4\pi\varepsilon_0=1$), so
> energies are in **Hartree** ($1\,\text{Ha}=27.211\,$eV) and lengths in **Bohr radii** ($a_0=1$, i.e.
> $0.529\,$Å). The radial Hamiltonian is **tridiagonal** (the $(1,-2,1)$ stencil), so we use
> `scipy.linalg.eigh_tridiagonal` rather than the dense `numpy.linalg.eigh` of [§6.10](schrodinger-on-a-computer.ipynb) — the dense solver
> is far too slow on the fine grids and many $l$ values this notebook needs. **Numerical honesty:** high-
> $n$ orbitals are large and diffuse ($\langle r\rangle\sim n^2 a_0$), so the box $r_{\max}$ must be big
> enough to contain them, or their energies come out spuriously *high* — a box artifact, not physics
> (the [§6.10](schrodinger-on-a-computer.ipynb)/[§6.11](bound-states-1d.ipynb) lesson). We keep comparisons to well-contained levels. The radial node count is
> $n_r=n-l-1$. See Sakurai & Napolitano and Griffiths (the hydrogen atom, the degeneracy, the Runge–Lenz
> vector); and Notebooks [§6.16](central-potentials-3d.ipynb) (the radial equation), [§6.15](orbital-angular-momentum.ipynb) (the spherical harmonics), [§6.10](schrodinger-on-a-computer.ipynb) (the
> eigenmethod), [§6.12](harmonic-oscillator.ipynb) (Laguerre/Hermite special functions).

## Theory in brief

### The Coulomb problem

The electron in a hydrogen-like atom feels the central potential

```{math}
:label: eq-hydrogen-coulomb
V(r)=-\frac{Ze^2}{4\pi\varepsilon_0 r}=-\frac{Z}{r}\quad(\text{atomic units}),\qquad V_{\text{eff}}(r)=-\frac{Z}{r}+\frac{l(l+1)}{2r^2} .
```

This is a central potential ([§6.16](central-potentials-3d.ipynb)), so the stationary states factor as $\psi_{n,l,m}=R_{n,l}(r)Y_l^m(
\theta,\varphi)$ and the radial function obeys the 1-D radial equation with the Coulomb well plus the
centrifugal barrier. Their competition — the $-Z/r$ attraction versus the $l(l+1)/r^2$ repulsion — sets
the sizes of the orbitals.

### The Rydberg spectrum

Solving the radial equation (a power-series analysis that Griffiths carries out in full) gives the
bound-state energies

```{math}
:label: eq-rydberg
E_n=-\frac{Z^2}{2n^2}\ \text{Hartree}=-\frac{13.6\,Z^2}{n^2}\ \text{eV},\qquad n=n_r+l+1=1,2,3,\dots ,
```

with $n$ the **principal** quantum number and $n_r$ the number of radial nodes. The *differences*
$E_n-E_{n'}$ are the spectral-line energies: the **Lyman** series (to $n=1$, ultraviolet), **Balmer** (to
$n=2$, visible), and **Paschen** (to $n=3$, infrared). This is the formula that explained the hydrogen
spectrum — quantum mechanics' first great empirical triumph.

### The three quantum numbers and the orbitals

The separation of variables in [§6.16](central-potentials-3d.ipynb) did most of the labelling for us: the angular equation fixed
$Y_l^m$ with its integers $l$ and $m$, and the radial equation contributes one more integer, the node
count $n_r$, which combines with $l$ into the principal quantum number $n$. Every bound state therefore
carries three labels:

```{math}
:label: eq-orbitals
\psi_{n,l,m}(r,\theta,\varphi)=R_{n,l}(r)\,Y_l^m(\theta,\varphi),\qquad 0\le l\le n-1,\quad -l\le m\le l,\quad n_r=n-l-1 .
```

The radial function $R_{n,l}$ (an exponential times an associated Laguerre polynomial) has $n_r=n-l-1$
nodes; the angular factor is a spherical harmonic ([§6.15](orbital-angular-momentum.ipynb)). Together they are the **atomic orbitals** —
1s, 2s, 2p, 3s, 3p, 3d, and so on. The radial density $r^2|R|^2$ gives the probability of finding the
electron at radius $r$; for the ground state it peaks at the **Bohr radius** $a_0$.

### The $l$-degeneracy and the hidden symmetry

In a generic central potential the energy depends on $n_r$ and $l$ separately, as the wells of [§6.16](central-potentials-3d.ipynb)
showed. For the Coulomb potential something remarkable happens: the energies {eq}`eq-rydberg` collapse
onto the single combination $n=n_r+l+1$, so that

```{math}
:label: eq-degeneracy
E_n\ \text{depends on } n\ \text{only, not on } l\ \Longrightarrow\ \text{all } l=0,\dots,n-1\ \text{share the energy } -Z^2/2n^2 .
```

Rotational symmetry alone guarantees only the $(2l+1)$-fold $m$-degeneracy; the extra $l$-degeneracy is
"accidental" — the fingerprint of a **hidden symmetry** special to $1/r$. The quantum **Runge–Lenz
vector** $\mathbf A$ is conserved (as it is classically, where it points along the fixed major axis of a
non-precessing Kepler ellipse), and together with $\mathbf L$ it generates the group $SO(4)$, larger than
$SO(3)$. This enlarged symmetry forces the $l$-degeneracy — the same phenomenon as the isotropic
oscillator's $2n_r+l$ degeneracy ([§6.16](central-potentials-3d.ipynb)), here in its sharpest form. Any departure from exact $1/r$ — the
screening of the nucleus by inner electrons in a many-electron atom — **breaks** the symmetry and splits
$s,p,d$, and *that splitting is why the periodic table has the structure it does*.

### Shell degeneracy and the periodic table

Counting the states of shell $n$ is now pure arithmetic: each allowed $l$ contributes its $2l+1$
orientations, and the $l$-degeneracy just established puts them all at one energy, so the sum runs over
$l=0,\dots,n-1$ and gives

```{math}
:label: eq-shells
\sum_{l=0}^{n-1}(2l+1)=n^2\ \text{orbitals per shell}\ \Longrightarrow\ 2n^2\ \text{with spin}=2,8,18,32,\dots
```

the exact row capacities of the periodic table. The shell structure of matter is the degeneracy counting
of the hydrogen atom (spin, the factor of 2, is added properly in [§6.18](spin-magnetic.ipynb)).

### Realism caveats

Before trusting the triumph, one should ask what the Hamiltonian left out. The answer, quantified by
the perturbative estimates that Griffiths works through systematically, is: nothing above the $10^{-3}$
level in relative terms, because every neglected effect is suppressed by powers of the fine-structure
constant $\alpha\approx1/137$ or of the mass ratio $m_e/m_p$. In summary,

```{math}
:label: eq-caveats
E_1=-13.6\,\text{eV}\ \text{is the gross structure; the corrections are}\ \lesssim10^{-3}\ \text{of it.}
```

This is the non-relativistic, infinite-nuclear-mass, spinless-Coulomb atom. Real hydrogen has small
corrections — the finite-mass (reduced-mass) shift, **fine structure** (relativistic + spin–orbit, [§6.21](perturbation-fine-structure.ipynb)),
the **Lamb shift** (QED), and **hyperfine** structure (the 21-cm line) — all tiny on the scale of
$-13.6\,$eV. The gross structure computed here is right to about $0.05\%$ (reduced-mass-limited).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import eigh_tridiagonal
from scipy.special import factorial, genlaguerre

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED, BLUE = "#c1121f", "#1d4e89"

# atomic units: ℏ = m_e = e = 4πε₀ = 1
HARTREE_EV = 27.211386  # 1 Hartree in electron-volts
BOHR_ANGSTROM = 0.529177  # the Bohr radius a₀ in ångströms
HC_EV_NM = 1239.841984  # hc in eV·nm (for wavelengths)


def solve_hydrogen_radial(l, Z, rmax, N):
    r"""Solve the hydrogen-like radial equation for angular momentum ``l`` and nuclear charge ``Z``.

    Reuses the §6.16 reduction $u=rR$ with the Coulomb effective potential $V_{\text{eff}}=-Z/r+l(l+1)/2r^2$
    (atomic units), on the interior grid $r\in(0,r_{\max})$. The radial Hamiltonian is **tridiagonal** —
    the $(1,-2,1)/dr^2$ kinetic stencil plus $\mathrm{diag}\,V_{\text{eff}}$ — so it is diagonalized with
    ``scipy.linalg.eigh_tridiagonal`` (far faster than the dense ``numpy.linalg.eigh`` of §6.10 on the fine
    grids used here). Eigenvectors are divided by $\sqrt{dr}$ so $\int|u|^2dr=1$.

    Parameters
    ----------
    l : int
        The orbital angular-momentum quantum number.
    Z : float
        The nuclear charge (``1`` for hydrogen).
    rmax : float
        The outer box radius in Bohr radii (must contain the orbital; high-$n$ states need a large box).
    N : int
        The number of interior grid points.

    Returns
    -------
    r : numpy.ndarray
        The radial grid (Bohr radii).
    energies : numpy.ndarray
        The radial energies (Hartree), ascending.
    u : numpy.ndarray
        Column $n_r$ is the normalized reduced radial function $u_{n_r,l}(r)=rR(r)$.
    """
    r = np.linspace(0.0, rmax, N + 2)[1:-1]  # interior grid: u(0)=u(rmax)=0
    dr = r[1] - r[0]
    Veff = -Z / r + l * (l + 1) / (2 * r**2)
    diag = 1.0 / dr**2 + Veff  # kinetic diagonal (−½·(−2)/dr²) + potential
    offdiag = -0.5 / dr**2 * np.ones(N - 1)  # kinetic off-diagonal (−½·1/dr²)
    energies, u = eigh_tridiagonal(diag, offdiag)
    return r, energies, u / np.sqrt(dr)


def hydrogen_radial_analytic(n, l, r, Z=1):
    r"""The exact hydrogen radial function $R_{n,l}(r)$, via ``scipy.special.genlaguerre``.

    $R_{n,l}(r)=N_{nl}\,e^{-Zr/n}(2Zr/n)^l L_{n-l-1}^{2l+1}(2Zr/n)$ with $L$ the associated Laguerre
    polynomial and $N_{nl}$ the normalization $\int|R|^2r^2dr=1$. Used to check the grid solutions
    {eq}`eq-orbitals`.
    """
    norm = np.sqrt((2 * Z / n) ** 3 * factorial(n - l - 1) / (2 * n * factorial(n + l)))
    rho = 2 * Z * r / n
    return norm * np.exp(-rho / 2) * rho**l * genlaguerre(n - l - 1, 2 * l + 1)(rho)


def radial_density(u):
    r"""The radial probability density $|u(r)|^2=r^2|R(r)|^2$ — the probability per unit $r$ (§6.16)."""
    return np.abs(u) ** 2


def real_spherical_harmonic(l, m, theta, phi):
    r"""The **real** spherical harmonic (the orbital angular shape), reused from §6.15 / Volume III."""
    from scipy.special import sph_harm_y

    if m == 0:
        return sph_harm_y(l, 0, theta, phi).real
    if m > 0:
        return np.sqrt(2.0) * (-1.0) ** m * sph_harm_y(l, m, theta, phi).real
    return np.sqrt(2.0) * (-1.0) ** m * sph_harm_y(l, -m, theta, phi).imag

## Exercise 1 — The Coulomb radial equation and the ground state

Solve the hydrogen radial equation for $l=0$ and find the ground-state energy, confirming
$E_1=-0.5\,$Ha $=-13.6\,$eV {eq}`eq-hydrogen-coulomb`, {eq}`eq-rydberg`.

1. Build $V_{\text{eff}}=-Z/r+l(l+1)/2r^2$ with $l=0$, $Z=1$ (no barrier for $s$ states).
2. Solve on the radial grid with the [§6.16](central-potentials-3d.ipynb) reduction and the tridiagonal solver
   (`solve_hydrogen_radial`, which uses `scipy.linalg.eigh_tridiagonal`).
3. Read the lowest energy.
4. Confirm $E_1=-0.5\,$Ha and convert to $-13.6\,$eV. Frame: the first exact atom.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    E1,
    -0.5,
    "the hydrogen ground state is E₁ = −0.5 Ha = −13.6 eV (the first exactly-solved atom)",
    atol=2e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The Rydberg spectrum

Compute the hydrogen spectrum for several $l$ and confirm the **Rydberg formula**
$E_n=-1/2n^2\,$Ha $=-13.6/n^2\,$eV, assigning each level its principal number $n=n_r+l+1$
{eq}`eq-rydberg`.

1. Solve for $l=0,1,2$, collecting the bound energies ($E<0$).
2. Assign each level its principal number $n=n_r+l+1$ ($n_r$ = its index within the fixed-$l$
   list).
3. Compare to $-1/2n^2$.
4. Confirm the Rydberg pattern for the well-contained levels (and note that high-$n$ levels need a
   bigger box — a box artifact, not physics). Frame: the formula that explained the spectral
   lines.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.array(E_grid),
    np.array(E_exact),
    "the hydrogen spectrum is Eₙ = −Z²/2n² Hartree (the Rydberg formula), for every (n,l)",
    atol=2e-3,
)

```{admonition} With your assistant
:class: tip
Ask your assistant for a radial-grid convergence sweep of its own design —
grid sizes, box radii, however it structures the loop — and run it. The gate
is the one this atom is famous for: the computed levels must approach
$E_n = -1/(2n^2)$ Hartree, and a sweep that "converges" anywhere else has
converged to its own discretization, not to hydrogen. The check is yours.
```

## Exercise 3 — Spectral lines: Lyman, Balmer, Paschen

Compute the wavelengths of the hydrogen spectral series from the energy differences, and identify
the visible **Balmer** lines ($H\alpha\approx656\,$nm) {eq}`eq-rydberg`.

1. From $E_n=-1/2n^2\,$Ha, form the transition energies $\Delta E=E_{n'}-E_n$ for $n'>n$.
2. Convert to wavelengths $\lambda=hc/\Delta E$ (with $hc=1239.84\,$eV·nm).
3. Identify the Balmer series ($n=2$, visible) — $H\alpha$ ($3\to2$), $H\beta$ ($4\to2$).
4. Compare to the observed lines. Frame: the theory reproduces spectroscopy, line for line.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    h_alpha,
    656.3,
    "the energy differences reproduce the hydrogen spectral series — the Balmer Hα line at 656 nm",
    atol=2.0,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The orbitals: radial functions and shapes

Compute the 1s, 2s, 2p (and 3d) radial functions, confirm they match the analytic forms with the
right node counts $n_r=n-l-1$, and build the full orbital $R_{n,l}Y_l^m$ {eq}`eq-orbitals`.

1. Extract $R_{n,l}(r)=u/r$ from the grid for $(1,0),(2,0),(2,1)$.
2. Compare to the analytic Laguerre forms (`hydrogen_radial_analytic`,
   `scipy.special.genlaguerre`).
3. Confirm the radial node count $n_r=n-l-1$ (1s: 0, 2s: 1, 2p: 0).
4. Plot $R_{n,l}$, the radial densities $r^2|R|^2$, and a cross-section of the full orbital
   density $|\psi_{n,l,m}|^2$ (reusing the [§6.15](orbital-angular-momentum.ipynb) angular shapes). Frame: the atomic orbitals,
   computed.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    match_ok,
    "the grid radial functions match the analytic hydrogen orbitals (scipy.special.genlaguerre) with n_r=n−l−1 nodes",
)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The most probable radius and the Bohr radius

Find the most probable distance of the electron from the nucleus in the ground state, and confirm
it is the **Bohr radius** $a_0$ {eq}`eq-orbitals`.

1. Form the ground-state radial density $r^2|R_{10}|^2=|u_{10}|^2$ (`radial_density`).
2. Find its maximum (`numpy.argmax`).
3. Confirm the peak is at $r=a_0$ ($=1$ in atomic units).
4. Contrast with the mean $\langle r\rangle=\int r\,|u|^2dr$, which is *larger* ($1.5\,a_0$)
   because the density has a long outward tail. Frame: the Bohr radius as the most probable
   radius, recovered from the wavefunction.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    r_peak,
    1.0,
    "the ground-state radial density peaks at the Bohr radius a₀ (=1 atomic unit)",
    atol=2e-2,
)

## Exercise 6 — The $l$-degeneracy and the hidden symmetry

Show that hydrogen levels of the same $n$ but different $l$ are **degenerate** — the energy
depends only on $n$ — and explain why this "accidental" degeneracy is the fingerprint of a hidden
symmetry {eq}`eq-degeneracy`.

1. Collect the energies of the $(n,l)$ states for $n=1..4$, $l=0..n-1$ (the lowest $l$-state has
   $n=l+1$).
2. Confirm all $l$ at fixed $n$ share the energy $-1/2n^2$.
3. Note rotational symmetry explains only the $m$-degeneracy, so the $l$-degeneracy is
   "accidental."
4. Explain it as the conserved quantum **Runge–Lenz vector** enlarging $SO(3)$ to $SO(4)$ — the
   closed Kepler orbit — and note that screening (any departure from $1/r$) splits $s,p,d$. Frame:
   the degeneracy is a symmetry fingerprint, and its breaking builds the periodic table.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    np.array(deg_grid),
    np.array(deg_exact),
    "hydrogen's energy depends only on n, not l — the SO(4) hidden-symmetry (Runge–Lenz) degeneracy",
    atol=2e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Shell degeneracy and the periodic table *(student)*

Count the orbitals in each shell and connect the count to the row capacities of the periodic table
{eq}`eq-shells`.

1. For $n=1..4$, count the orbitals $\sum_{l=0}^{n-1}(2l+1)$ and check it equals $n^2$.
2. Multiply by 2 for the two spin states ([§6.18](spin-magnetic.ipynb)) to get $2n^2$.
3. Compare to the periodic table's row lengths $2,8,18,32$.
4. Note the shell structure of matter *is* hydrogen's degeneracy counting (with the caveat that
   screening reorders subshells in real atoms). Frame: the atom's degeneracies are the
   architecture of chemistry.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    counts_ok,
    "shell n holds n² orbitals (2n² with spin) = the periodic-table row capacities 2,8,18,32 — the n² degeneracy underlies the periodic table",
)

## Exercise 8 — Hydrogen-like ions and scaling *(student)*

Solve a hydrogen-like ion (He$^+$, Li$^{2+}$, …) and find how the energies and sizes scale with
the nuclear charge $Z$ {eq}`eq-rydberg`, {eq}`eq-hydrogen-coulomb`.

1. Solve `solve_hydrogen_radial` with nuclear charge $Z=1,2,3$.
2. Confirm $E_n=-Z^2/2n^2$ (energies scale as $Z^2$).
3. Confirm the orbitals shrink as $1/Z$ (the ground-state peak moves to $a_0/Z$).
4. Interpret: a stronger nucleus binds tighter and pulls the electron in. Frame: one solution, a
   whole isoelectronic family.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    np.array(scale_grid),
    np.array(scale_exact),
    "hydrogen-like ion energies scale as Eₙ = −Z²/2n² and orbitals shrink as 1/Z",
    rtol=1e-2,
)

## Exercise 9 — The atom that built the theory

Everything in this volume came together here. The spherical harmonics gave us the angles ([§6.15](orbital-angular-momentum.ipynb)), the
radial reduction gave us the distance ([§6.16](central-potentials-3d.ipynb)), and the eigenvalue solver gave us the energies ([§6.10](schrodinger-on-a-computer.ipynb)) — and
out of the three came the hydrogen atom: the Rydberg spectrum that matched the spectral lines to a
fraction of a percent, the orbitals that shape every molecule, and a degeneracy so exact it revealed a
symmetry no one had put in by hand.

**This exercise (synthesis).** No new computation: the atom is the result. The energy's blindness to $l$
is the quantum echo of the Kepler ellipse that never precesses — the conserved Runge–Lenz vector, the
same $SO(4)$ symmetry we first met as the *closed orbit* of an inverse-square force in Volumes I and II,
and whose tiny violation (Mercury's precession) opened the door to general relativity in Volume IV. Here
it is exact, and its *breaking* — by letting inner electrons screen the nucleus, so the potential is no
longer quite $1/r$ — is what splits $s$ from $p$ from $d$ and gives the periodic table its rows and
columns. What remains is to add the one thing the Schrödinger equation left out: the electron's **spin**,
the fourth quantum number, which the next notebook ([§6.18](spin-magnetic.ipynb)) restores and which completes the $2n^2$ counting
— and which fine structure ([§6.21](perturbation-fine-structure.ipynb)) will use to split these perfect Coulomb levels at last.

Bohr guessed the $-13.6\,$eV$/n^2$ formula in 1913 from a model he knew was provisional. Thirteen years
later the Schrödinger equation produced it *exactly*, along with the orbitals Bohr could not have drawn,
and hid inside it a symmetry that took another decade to name. We computed the whole of it, on a grid, in
an afternoon.

## Notebook summary

The hydrogen atom — the crown of Movement III, and a summit of the volume.

- **The Coulomb problem** {eq}`eq-hydrogen-coulomb`: $V=-Z/r$ in the [§6.16](central-potentials-3d.ipynb) radial equation, $V_{\text{eff}}=-Z/r+
  l(l+1)/2r^2$, solved with `scipy.linalg.eigh_tridiagonal` in atomic units.
- **The Rydberg spectrum** {eq}`eq-rydberg`: $E_n=-Z^2/2n^2\,$Ha $=-13.6\,Z^2/n^2\,$eV; the differences
  are the Lyman/Balmer/Paschen lines ($H\alpha=656\,$nm), matched to spectroscopy.
- **The orbitals** {eq}`eq-orbitals`: $\psi_{n,l,m}=R_{n,l}Y_l^m$, radial nodes $n_r=n-l-1$, matching the
  analytic Laguerre forms to $\sim10^{-4}$; the 1s density peaks at the Bohr radius $a_0$.
- **The $l$-degeneracy** {eq}`eq-degeneracy`: $E$ depends on $n$ only — the Runge–Lenz / $SO(4)$ hidden
  symmetry (the closed Kepler orbit), broken by screening in many-electron atoms.
- **Shell degeneracy** {eq}`eq-shells`: $n^2$ orbitals ($2n^2$ with spin) $=2,8,18,32$ — the periodic
  table's rows.
- **Scaling**: hydrogen-like ions have $E\propto Z^2$ and sizes $\propto1/Z$.

The first exact atom, the spectrum that explained the lines, and a hidden symmetry linking the quantum
atom to the classical Kepler orbit. Only spin is missing — and that is next.

## Outlook

- **Spin ([§6.18](spin-magnetic.ipynb))**: the fourth quantum number $m_s$, the electron's intrinsic angular momentum (the
  half-integers of [§6.14](angular-momentum-algebra.ipynb)), completing the $2n^2$ counting.
- **Fine structure ([§6.21](perturbation-fine-structure.ipynb))**: relativistic and spin–orbit corrections lift the $l$-degeneracy; the Lamb
  shift (QED) and hyperfine structure (the 21-cm line) are further, tinier splittings (horizons).
- **Many-electron atoms and the periodic table** (a horizon): screening breaks the $SO(4)$ symmetry and
  orders the subshells, with the exclusion principle filling them.
- **The Runge–Lenz vector and $SO(4)$** (an algebraic horizon): hydrogen solved by symmetry alone, with no
  differential equation — the Pauli/Fock method.
- **Cross-reference** [§6.16](central-potentials-3d.ipynb) (the radial equation), [§6.15](orbital-angular-momentum.ipynb) (the spherical harmonics), [§6.10](schrodinger-on-a-computer.ipynb) (the eigenmethod),
  [§6.12](harmonic-oscillator.ipynb) (Laguerre/Hermite special functions), and forward to [§6.18](spin-magnetic.ipynb), [§6.21](perturbation-fine-structure.ipynb).

In [ ]:
from ecp.style import footer

footer()